# imports

In [20]:
from pathlib import Path
import ast
import json
import time
import copy

import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import f1_score

# raíz del proyecto y device

In [21]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("No pude encontrar la raíz del proyecto.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
WORKING_DIR = DATA_DIR / "working"
MANIFESTS_DIR = DATA_DIR / "manifests"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "nih_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2
OUTPUT_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/nih_baseline
DEVICE: cuda
GPU: NVIDIA GeForce RTX 4060


# cargar subset y label map

In [22]:
subset_path = WORKING_DIR / "nih" / "subsets" / "nih_subset_large_with_paths.csv"
manifest_full_path = MANIFESTS_DIR / "manifest_nih_final_with_paths.csv"

subset_df = pd.read_csv(subset_path)
manifest_full = pd.read_csv(manifest_full_path)

print("subset_df:", subset_df.shape)
print("manifest_full:", manifest_full.shape)
print("\nConteo por split:")
print(subset_df["split_final"].value_counts())

subset_df: (19000, 19)
manifest_full: (112120, 19)

Conteo por split:
split_final
train    15000
val       2000
test      2000
Name: count, dtype: int64


# parsear labels

In [23]:
def safe_parse_labels(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    x = str(x).strip()
    if not x:
        return []
    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass
    return [label.strip() for label in x.split("|") if label.strip()]

def clean_labels(labels):
    return [label for label in labels if label != "No Finding"]

subset_df["labels_list"] = subset_df["labels_list"].apply(safe_parse_labels).apply(clean_labels)
manifest_full["labels_list"] = manifest_full["labels_list"].apply(safe_parse_labels).apply(clean_labels)

subset_df[["image_name", "labels_list"]].head()

,image_name,labels_list
0,00022245_021.png,[]
1,00019544_000.png,[]
2,00009673_001.png,[Pleural_Thickening]
3,00018103_001.png,[]
4,00017799_000.png,[Nodule]


# separar train / val / test

In [24]:
train_df = subset_df[subset_df["split_final"] == "train"].copy()
val_df = subset_df[subset_df["split_final"] == "val"].copy()
test_df = subset_df[subset_df["split_final"] == "test"].copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (15000, 19)
val: (2000, 19)
test: (2000, 19)


## prueba esta es la 5.5 celda


In [25]:
all_labels = sorted({
    label
    for labels in manifest_full["labels_list"]
    for label in labels
})

label_to_idx = {label: i for i, label in enumerate(all_labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

print("Etiquetas finales usadas:")
print(all_labels)
print("\nNúmero de etiquetas:", len(all_labels))

Etiquetas finales usadas:
['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']

Número de etiquetas: 14


# función multihot

In [26]:
def labels_to_multihot(labels, label_to_idx):
    vec = torch.zeros(len(label_to_idx), dtype=torch.float32)
    for label in labels:
        if label in label_to_idx:
            vec[label_to_idx[label]] = 1.0
    return vec

## esta tambien es de prueba 6.5 celda

In [27]:
# Construimos matriz multihot del train
train_targets = torch.stack([
    labels_to_multihot(labels, label_to_idx)
    for labels in train_df["labels_list"]
])

pos_counts = train_targets.sum(dim=0)
neg_counts = len(train_targets) - pos_counts

# Evitamos división por cero
pos_weight = neg_counts / torch.clamp(pos_counts, min=1.0)

print("pos_counts:", pos_counts)
print("pos_weight:", pos_weight)

pos_counts: tensor([1423.,  311.,  488.,  240., 1493.,  223.,  211.,   22., 2354.,  681.,
         781.,  370.,  150.,  451.])
pos_weight: tensor([  9.5411,  47.2315,  29.7377,  61.5000,   9.0469,  66.2646,  70.0900,
        680.8182,   5.3721,  21.0264,  18.2061,  39.5405,  99.0000,  32.2594])


# transforms

In [28]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# dataset

In [29]:
class NIHSubsetDataset(Dataset):
    def __init__(self, dataframe, label_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["file_path"])
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        target = labels_to_multihot(row["labels_list"], self.label_to_idx)

        return {
            "image": image,
            "target": target,
            "image_name": row["image_name"],
        }

# datasets y dataloaders

In [30]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset = NIHSubsetDataset(train_df, label_to_idx, transform=train_transform)
val_dataset = NIHSubsetDataset(val_df, label_to_idx, transform=eval_transform)
test_dataset = NIHSubsetDataset(test_df, label_to_idx, transform=eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Datasets y dataloaders listos.")

Datasets y dataloaders listos.


# modelo baseline

In [31]:
NUM_CLASSES = len(label_to_idx)

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

print(model.fc)
print("Número de clases:", NUM_CLASSES)

Linear(in_features=512, out_features=14, bias=True)
Número de clases: 14


In [32]:
def freeze_backbone(model):
    for name, param in model.named_parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True

freeze_backbone(model)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Parámetros entrenables:", trainable_params)
print("Parámetros totales:", total_params)

Parámetros entrenables: 7182
Parámetros totales: 11183694


# loss, optimizer y scheduler

In [33]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

In [34]:
def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

unfreeze_lr = 1e-4

# función de evaluación

In [35]:
def evaluate_model(model, loader, criterion, device, threshold=0.3):
    model.eval()

    total_loss = 0.0
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            targets = batch["target"].to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, targets)

            probs = torch.sigmoid(logits)

            total_loss += loss.item() * images.size(0)
            all_targets.append(targets.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    all_targets = np.concatenate(all_targets, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = (all_probs >= threshold).astype(int)

    f1_macro = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    f1_micro = f1_score(all_targets, all_preds, average="micro", zero_division=0)

    return avg_loss, f1_macro, f1_micro

# loop de entrenamiento

In [36]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss

# entrenamiento completo

In [37]:
EARLY_STOPPING_PATIENCE = 4

history = []
best_val_loss = float("inf")
best_model_state = copy.deepcopy(model.state_dict())
best_epoch = 0
epochs_without_improvement = 0

PHASE1_EPOCHS = 4
PHASE2_EPOCHS = 12
current_epoch = 0

print("=== FASE 1: entrenamiento de la cabeza ===")

for local_epoch in range(1, PHASE1_EPOCHS + 1):
    current_epoch += 1
    start_time = time.time()

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_f1_macro, val_f1_micro = evaluate_model(model, val_loader, criterion, DEVICE)

    scheduler.step(val_loss)
    elapsed = time.time() - start_time

    history.append({
        "epoch": current_epoch,
        "phase": 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_f1_macro": val_f1_macro,
        "val_f1_micro": val_f1_micro,
        "lr": optimizer.param_groups[0]["lr"],
        "time_sec": elapsed,
    })

    print(
        f"[F1] Epoch {local_epoch:02d}/{PHASE1_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1_macro={val_f1_macro:.4f} | "
        f"val_f1_micro={val_f1_micro:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.6f} | "
        f"time={elapsed:.1f}s"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = current_epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping activado en fase 1, época global {current_epoch}.")
        break

print("\n=== FASE 2: fine-tuning completo ===")

unfreeze_all(model)

optimizer = optim.Adam(model.parameters(), lr=unfreeze_lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

for local_epoch in range(1, PHASE2_EPOCHS + 1):
    current_epoch += 1
    start_time = time.time()

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_f1_macro, val_f1_micro = evaluate_model(model, val_loader, criterion, DEVICE)

    scheduler.step(val_loss)
    elapsed = time.time() - start_time

    history.append({
        "epoch": current_epoch,
        "phase": 2,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_f1_macro": val_f1_macro,
        "val_f1_micro": val_f1_micro,
        "lr": optimizer.param_groups[0]["lr"],
        "time_sec": elapsed,
    })

    print(
        f"[F2] Epoch {local_epoch:02d}/{PHASE2_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1_macro={val_f1_macro:.4f} | "
        f"val_f1_micro={val_f1_micro:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.6f} | "
        f"time={elapsed:.1f}s"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = current_epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping activado en fase 2, época global {current_epoch}.")
        break

print(f"\nEntrenamiento terminado. Mejor época global: {best_epoch} | best_val_loss={best_val_loss:.4f}")

=== FASE 1: entrenamiento de la cabeza ===
[F1] Epoch 01/4 | train_loss=1.3559 | val_loss=1.3537 | val_f1_macro=0.1015 | val_f1_micro=0.1065 | lr=0.001000 | time=510.0s
[F1] Epoch 02/4 | train_loss=1.2256 | val_loss=1.3920 | val_f1_macro=0.1116 | val_f1_micro=0.1166 | lr=0.001000 | time=242.3s
[F1] Epoch 03/4 | train_loss=1.1865 | val_loss=1.3336 | val_f1_macro=0.1104 | val_f1_micro=0.1202 | lr=0.001000 | time=246.3s
[F1] Epoch 04/4 | train_loss=1.1697 | val_loss=1.3193 | val_f1_macro=0.1074 | val_f1_micro=0.1122 | lr=0.001000 | time=240.8s

=== FASE 2: fine-tuning completo ===
[F2] Epoch 01/12 | train_loss=1.1875 | val_loss=1.6185 | val_f1_macro=0.1139 | val_f1_micro=0.1332 | lr=0.000100 | time=249.4s
[F2] Epoch 02/12 | train_loss=1.0502 | val_loss=1.3213 | val_f1_macro=0.1096 | val_f1_micro=0.1247 | lr=0.000100 | time=244.1s
[F2] Epoch 03/12 | train_loss=0.9648 | val_loss=1.2611 | val_f1_macro=0.1320 | val_f1_micro=0.1522 | lr=0.000100 | time=243.9s
[F2] Epoch 04/12 | train_loss=0.86

In [38]:
def collect_probs_targets(model, loader, device):
    model.eval()
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            targets = batch["target"].to(device, non_blocking=True)

            logits = model(images)
            probs = torch.sigmoid(logits)

            all_targets.append(targets.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    all_targets = np.concatenate(all_targets, axis=0)
    all_probs = np.concatenate(all_probs, axis=0)
    return all_targets, all_probs

model.load_state_dict(best_model_state)

val_targets, val_probs = collect_probs_targets(model, val_loader, DEVICE)

candidate_thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
threshold_results = []

for thr in candidate_thresholds:
    val_preds = (val_probs >= thr).astype(int)
    val_f1_macro = f1_score(val_targets, val_preds, average="macro", zero_division=0)
    val_f1_micro = f1_score(val_targets, val_preds, average="micro", zero_division=0)
    threshold_results.append((thr, val_f1_macro, val_f1_micro))

threshold_df = pd.DataFrame(
    threshold_results,
    columns=["threshold", "val_f1_macro", "val_f1_micro"]
)

display(threshold_df)

best_threshold = threshold_df.sort_values("val_f1_macro", ascending=False).iloc[0]["threshold"]
print("Best threshold según val_f1_macro:", best_threshold)

,threshold,val_f1_macro,val_f1_micro
0,0.1,0.104911,0.116289
1,0.2,0.117830,0.133460
2,0.3,0.132011,0.152191
3,0.4,0.144870,0.169977
4,0.5,0.155625,0.184108


Best threshold según val_f1_macro: 0.5


# cargar mejor modelo y evaluar en test

In [39]:
model.load_state_dict(best_model_state)

test_loss, _, _ = evaluate_model(model, test_loader, criterion, DEVICE, threshold=best_threshold)

test_targets, test_probs = collect_probs_targets(model, test_loader, DEVICE)
test_preds = (test_probs >= best_threshold).astype(int)

test_f1_macro = f1_score(test_targets, test_preds, average="macro", zero_division=0)
test_f1_micro = f1_score(test_targets, test_preds, average="micro", zero_division=0)

print("Resultados en test:")
print(f"threshold usado = {best_threshold:.2f}")
print(f"test_loss       = {test_loss:.4f}")
print(f"test_f1_macro   = {test_f1_macro:.4f}")
print(f"test_f1_micro   = {test_f1_micro:.4f}")

Resultados en test:
threshold usado = 0.50
test_loss       = 1.6623
test_f1_macro   = 0.1982
test_f1_micro   = 0.2352


# guardar historial y checkpoint

In [40]:
history_df = pd.DataFrame(history)
history_path = OUTPUT_DIR / "nih_baseline_history.csv"
history_df.to_csv(history_path, index=False)

checkpoint_path = OUTPUT_DIR / "nih_baseline_resnet18_subset_df.pth"
torch.save({
    "model_state_dict": best_model_state,
    "label_to_idx": label_to_idx,
    "image_size": IMAGE_SIZE,
    "num_classes": NUM_CLASSES,
    "best_val_loss": best_val_loss,
    "test_loss": test_loss,
    "test_f1_macro": test_f1_macro,
    "test_f1_micro": test_f1_micro,
}, checkpoint_path)

print("Historial guardado en:", history_path)
print("Checkpoint guardado en:", checkpoint_path)

Historial guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/nih_baseline/nih_baseline_history.csv
Checkpoint guardado en: /mnt/d/Universidad/analitica/proyecto_analitica2/outputs/nih_baseline/nih_baseline_resnet18_subset_df.pth


In [41]:
history_df

,epoch,phase,train_loss,val_loss,val_f1_macro,val_f1_micro,lr,time_sec
0,1,1,1.355865,1.353693,0.101476,0.106484,0.00100,509.988921
1,2,1,1.225641,1.392023,0.111630,0.116608,0.00100,242.305474
2,3,1,1.186482,1.333562,0.110389,0.120231,0.00100,246.276035
3,4,1,1.169662,1.319295,0.107396,0.112194,0.00100,240.844524
4,5,2,1.187488,1.618462,0.113926,0.133218,0.00010,249.388648
5,6,2,1.050225,1.321336,0.109634,0.124731,0.00010,244.088436
6,7,2,0.964794,1.261075,0.132011,0.152191,0.00010,243.934870
7,8,2,0.868850,1.382195,0.138810,0.167546,0.00010,247.777566
8,9,2,0.760907,1.417931,0.137683,0.176385,0.00010,248.024506
9,10,2,0.708591,1.646888,0.148878,0.171519,0.00005,242.087171
